## Similarity Metrics Selection

- *Objective*: Choose and justify the similarity metrics used to compare product vectors.
- *Key Actions*: Evaluate different metrics (e.g., cosine similarity, dot product) and select the best fit based on the dataset characteristics.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
import matplotlib.pyplot as plt
import seaborn as sns
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer


project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

from src.initialization import load_environment
load_environment()

In [ ]:
api_key = os.getenv('PINECONE_API_KEY')
pc = Pinecone(api_key=api_key)
index = pc.Index("product-vectors")

cleaned_df = pd.read_csv('../src/data/dataset/cleaned/cleaned_cnn.csv')

model = SentenceTransformer('all-MiniLM-L6-v2')

query_vector = model.encode("product")
results = index.query(
    vector=query_vector.tolist(),
    top_k=1000,
    include_metadata=True,
    include_values=True
)

In [ ]:
# Extract embeddings and metadata
embeddings = np.array([match.values for match in results.matches])  
descriptions = [match.metadata['Description'] for match in results.matches]

# Calculate different similarity metrics
cosine_sim = cosine_similarity(embeddings)
euclidean_dist = euclidean_distances(embeddings)

In [ ]:
# Visualize the results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.heatmap(cosine_sim[:10, :10], cmap='viridis')
plt.title('Cosine Similarity Matrix (First 10 Products)')

plt.subplot(1, 2, 2)
sns.heatmap(euclidean_dist[:10, :10], cmap='viridis')
plt.title('Euclidean Distance Matrix (First 10 Products)')

plt.tight_layout()
plt.show()

In [ ]:
# Print some example comparisons
display("\nExample Product Similarities:")
for i in range(3):
    display(f"\nProduct: {descriptions[i]}")
    similar_indices = cosine_sim[i].argsort()[-4:-1][::-1]
    for idx in similar_indices:
        display(f"Similar to: {descriptions[idx]} (Score: {cosine_sim[i][idx]:.3f})")